In [1]:
Path = './Test/test'

In [2]:
import os

In [3]:
import os

Path = "./test"

if not os.path.exists(Path):
    print("Path does not exist:", Path)
else:
    for label in os.listdir(Path):
        label_path = os.path.join(Path, label)
        print("Reading:", label_path)

Reading: ./test/DC Sneak Peek
Reading: ./test/Hawk and Dove
Reading: ./test/Wonder Woman
Reading: ./test/Batman Incorporated v2
Reading: ./test/Blackhawks
Reading: ./test/Damian - Son of Batman
Reading: ./test/Sinestro
Reading: ./test/Forever Evil - Rogues Rebellion
Reading: ./test/Dial H
Reading: ./test/Bat-Mite
Reading: ./test/Arkham Manor
Reading: ./test/Grifter v3
Reading: ./test/2017.11.01 Marvel Week+ GetComics.INFO
Reading: ./test/.DS_Store
Reading: ./test/Superboy
Reading: ./test/Red Hood and the Outlaws
Reading: ./test/Harley Quinn
Reading: ./test/We Are Robin
Reading: ./test/Forever Evil
Reading: ./test/Futures End Month
Reading: ./test/Black Canary v3
Reading: ./test/Huntress
Reading: ./test/Gotham by Midnight
Reading: ./test/Constantine - The Hellblazer
Reading: ./test/The Savage Hawkman v1
Reading: ./test/Harley Quinn and Power Girl
Reading: ./test/Justice League Dark
Reading: ./test/Human Bomb
Reading: ./test/Captain Atom
Reading: ./test/Variant Covers
Reading: ./test/Gra

In [4]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

# Original dataset path
source_path = "./test"

# New dataset path
output_path = "./dataset"

batman_folders = [
    "Batman v2",
    "Batman Eternal",
    "Batman & Robin Eternal",
    "Batman and Robin v2",
    "Batman Beyond v6",
    "Batman Incorporated v2",
    "Detective Comics v2",
    "Batgirl v4",
    "Batwoman",
    "Batwing",
    "Nightwing v3",
    "Red Hood and the Outlaws",
    "Damian - Son of Batman",
    "Arkham Manor",
    "Gotham Academy",
    "Gotham by Midnight",
    "We Are Robin",
    "Bat-Mite"
]

ignore_folders = [
    ".DS_Store",
    "X-Men Hidden Years",
    "2017.11.01 Marvel Week+ GetComics.INFO",
    "Variant Covers",
    "orj"
]

# Create output directories
for split in ["train", "test"]:
    for cls in ["Batman_Universe", "Other_DC"]:
        os.makedirs(os.path.join(output_path, split, cls), exist_ok=True)

def collect_images(folder_list, label_name):
    images = []
    for folder in folder_list:
        folder_path = os.path.join(source_path, folder)
        if os.path.exists(folder_path):
            for file in os.listdir(folder_path):
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    images.append(os.path.join(folder_path, file))
    return images

# Collect Batman images
batman_images = collect_images(batman_folders, "Batman_Universe")

# Collect Other DC images
other_folders = [f for f in os.listdir(source_path)
                 if f not in batman_folders and f not in ignore_folders]

other_images = collect_images(other_folders, "Other_DC")

# Train-test split
bat_train, bat_test = train_test_split(batman_images, test_size=0.2, random_state=42)
oth_train, oth_test = train_test_split(other_images, test_size=0.2, random_state=42)

def copy_images(image_list, split, class_name):
    for img in image_list:
        shutil.copy(img, os.path.join(output_path, split, class_name))

copy_images(bat_train, "train", "Batman_Universe")
copy_images(bat_test, "test", "Batman_Universe")

copy_images(oth_train, "train", "Other_DC")
copy_images(oth_test, "test", "Other_DC")

print("Dataset split complete.")

Dataset split complete.


In [5]:
import os

train_path = "./dataset/train"
test_path = "./dataset/test"

def count_images(path):
    counts = {}
    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
    return counts

print("Train distribution:", count_images(train_path))
print("Test distribution:", count_images(test_path))

Train distribution: {'Batman_Universe': 2180, 'Other_DC': 5979}
Test distribution: {'Batman_Universe': 545, 'Other_DC': 1500}


In [6]:
from PIL import Image
import numpy as np

def analyze_image_sizes(path):
    widths = []
    heights = []

    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        for file in os.listdir(cls_path):
            if file.lower().endswith(('.jpg','.png','.jpeg')):
                img = Image.open(os.path.join(cls_path, file))
                w, h = img.size
                widths.append(w)
                heights.append(h)

    print("Min Width:", min(widths))
    print("Max Width:", max(widths))
    print("Min Height:", min(heights))
    print("Max Height:", max(heights))
    print("Average Width:", np.mean(widths))
    print("Average Height:", np.mean(heights))

analyze_image_sizes(train_path)

Min Width: 288
Max Width: 288
Min Height: 432
Max Height: 432
Average Width: 288.0
Average Height: 432.0


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import os

In [8]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [9]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [10]:
train_dir = "./dataset/train"
test_dir="./dataset/test"

train_dataset = datasets.ImageFolder(train_dir,transform=train_transforms)
test_dataset = datasets.ImageFolder(test_dir,transform=test_transforms)

In [11]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 8159
Test size: 2045


In [12]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN,self).__init__()

        self.features = nn.Sequential(
            #Block 1 
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            #Block 2 
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [13]:
model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [14]:
train_counts = torch.tensor([2180,5979],dtype=torch.float)

total_samples = train_counts.sum()
num_classes = len(train_counts)

class_weights = total_samples / (num_classes * train_counts)
class_weights = class_weights.to(device)

print(class_weights)

tensor([1.8713, 0.6823], device='mps:0')


In [15]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [16]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
def train_one_epoch(model,loader,optimizer,criterion):
    model.train()
    running_loss = 0.0
    correct =0
    total=0

    for images,labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs,labels)

        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

        _,predicted = torch.max(outputs,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct/total

    return epoch_loss,epoch_acc

In [18]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [19]:
num_epochs = 15
best_val_acc = 0

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, test_loader, criterion)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print("-" * 40)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

KeyboardInterrupt: 